### 1. 패키지 설치 + 환경변수 로드

In [1]:
%pip install -qU langchain langchain_openai langgraph tavily

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 2. IN-Memory 설정

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()   # 그래프 State를 스텝마다 저장하는 체크포인터 (RAM 저장 → 프로세스 종료 시 소멸)

### 3. 도구(Tool) 정의

In [4]:
from langchain_core.tools import tool
from tavily import TavilyClient

client = TavilyClient()

@tool
def search_web(query: str) -> list[dict]:
    """웹에서 최신 정보나 외부 자료를 검색합니다"""

    response = client.search(
        query=query,
        search_depth="basic",
        max_results=5,
        include_raw_content=False
    )

    return response.get("results", [])

### 4. State와 LLM 정의

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class State(TypedDict):
    messages: Annotated[list, add_messages]   # add_messages reducer로 메시지를 누적 (덮어쓰지 않음)

tools = [search_web]   # 그래프에서 사용할 도구 목록

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)   # 도구 스키마를 LLM에 바인딩 → 필요 시 tool_calls 생성

def chatbot(state: State):   # 누적된 messages를 LLM에 전달, 응답을 리스트로 반환 → add_messages가 누적
    answer = llm_with_tools.invoke(state["messages"])
    return {"messages": [answer]}

### 5. 그래프(Graph) 구성

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))   # tool_calls를 실행해 ToolMessage로 반환하는 prebuilt 노드

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)   # tool_calls가 있으면 "tools", 없으면 END
graph_builder.add_edge("tools", "chatbot")   # 도구 결과를 다시 LLM에 전달 (chatbot ⇄ tools 루프)

### 6. 메모리 연결하여 컴파일

In [7]:
graph = graph_builder.compile(checkpointer=memory)   # 2번에서 만든 InMemorySaver를 연결 → 대화 상태 저장/복원

### 7. thread_id 설정 및 실행

In [8]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "1"}}   # 이 thread_id 기준으로 대화가 누적됨

question = "내 이름은 길동이야. 기억해줘"

for event in graph.stream(
    {"messages": [("user", question)]},
    config   # ← config를 넘겨야 checkpointer가 상태를 저장함
):
    for key, value in event.items():
        print(f"\n{'=' * 50}")
        print(f"STEP: {key}")
        print(f"{'=' * 50}")
        print(value["messages"][-1])


STEP: chatbot
content='길동님, 반가워요! 제가 정보를 기억할 수는 없지만, 이 대화 동안에는 길동님을 기억할 수 있어요. 무엇을 도와드릴까요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 60, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a17b1ca31d', 'id': 'chatcmpl-E6G3ZmzySIuXqlyxuctenqHxf8ux8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fa3d9-f2cb-7081-8fb9-21fe8a0e74c5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 60, 'output_tokens': 42, 'total_tokens': 102, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 8. 대화 기억 확인

In [9]:
question = "내 이름이 뭐라고 했지?"

for event in graph.stream(
    {"messages": [("user", question)]},
    config   # 같은 thread_id → 앞의 대화를 이어받아 이름을 기억함
):
    for key, value in event.items():
        print(f"\n{'=' * 50}")
        print(f"STEP: {key}")
        print(f"{'=' * 50}")
        print(value["messages"][-1])


STEP: chatbot
content='길동님이라고 하셨습니다!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 117, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a17b1ca31d', 'id': 'chatcmpl-E6G3e2Jwi5qDyDzO1uhvGBg4ZpD80', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fa3da-0b88-7501-a1de-527694dfdcc5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 117, 'output_tokens': 9, 'total_tokens': 126, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 9. 저장된 상태(State) 확인

In [10]:
snapshot = graph.get_state(config)   # checkpointer에 저장된 현재 thread의 상태 조회

print(snapshot.values["messages"])   # 누적된 전체 메시지
print("\n다음 실행 노드:", snapshot.next)   # 비어있으면 대기 완료 상태

[HumanMessage(content='내 이름은 길동이야. 기억해줘', additional_kwargs={}, response_metadata={}, id='505c89f7-6bda-44d6-83cc-1d84532b0ea0'), AIMessage(content='길동님, 반가워요! 제가 정보를 기억할 수는 없지만, 이 대화 동안에는 길동님을 기억할 수 있어요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 60, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a17b1ca31d', 'id': 'chatcmpl-E6G3ZmzySIuXqlyxuctenqHxf8ux8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa3d9-f2cb-7081-8fb9-21fe8a0e74c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 42, 'total_tokens': 102, 'input_token_